In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# CELL 1: Verify competition data
# ============================================================
import os, glob

COMP_DIR    = '/kaggle/input/competitions/stanford-rna-3d-folding-2'
PDB_RNA_DIR = os.path.join(COMP_DIR, 'PDB_RNA')
TEST_CSV    = os.path.join(COMP_DIR, 'test_sequences.csv')
FASTA_DB    = os.path.join(PDB_RNA_DIR, 'pdb_seqres_NA.fasta')
DATES_CSV   = os.path.join(PDB_RNA_DIR, 'pdb_release_dates_NA.csv')

WORK_DIR = '/kaggle/working'

print('=== Data verification ===')
for label, path in [
    ('Competition dir', COMP_DIR),
    ('PDB_RNA dir',     PDB_RNA_DIR),
    ('test_sequences',  TEST_CSV),
    ('FASTA database',  FASTA_DB),
    ('Release dates',   DATES_CSV),
]:
    exists = os.path.exists(path)
    print(f'  {label}: {"OK" if exists else "MISSING"} — {path}')

cif_files = glob.glob(os.path.join(PDB_RNA_DIR, '*.cif*'))
print(f'\nCIF files: {len(cif_files):,}')

if len(cif_files) == 0:
    print('\nERROR: No CIF files found.')
    print('Make sure stanford-rna-3d-folding-2 competition data is attached.')
    print('Click "+ Add Input" in the sidebar → search "stanford-rna-3d-folding-2"')

In [ ]:
# ============================================================
# CELL 2: Load and inspect test sequences
# ============================================================
import pandas as pd

test_df = pd.read_csv(TEST_CSV)
print(f'Test targets: {len(test_df)}')
print(f'Columns: {list(test_df.columns)}')
print(f'Sequence length range: {test_df["sequence"].str.len().min()} — {test_df["sequence"].str.len().max()}')
print()
print(test_df[['target_id', 'sequence']].head())

In [ ]:
# ============================================================
# CELL 3: Install dependencies
# ============================================================
# MMseqs2: fast sequence search (like BLAST but much faster)
#   Official: https://github.com/soedinglab/MMseqs2
#
# BioPython: for parsing CIF structure files
#   Official: https://biopython.org/
#
# NOTE: For final submission with internet OFF, you need to
# pre-install MMseqs2 as a Kaggle Dataset. See:
#   https://www.kaggle.com/datasets/rhijudas/mmseqs2-binary
# ============================================================
import subprocess, shutil

# Check if mmseqs is already available
if shutil.which('mmseqs') is None:
    print('Installing MMseqs2...')
    subprocess.run(['apt-get', 'update', '-qq'], check=False,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'mmseqs2'],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('MMseqs2 installed.')
else:
    print('MMseqs2 already available.')

!mmseqs version

# BioPython (usually pre-installed on Kaggle, but install if missing)
try:
    import Bio
    print(f'BioPython {Bio.__version__} available.')
except ImportError:
    !pip install biopython --quiet
    print('BioPython installed.')

In [ ]:
# ============================================================
# CELL 4: Clone official DasLab create_templates repository
# ============================================================
# Source: https://github.com/DasLab/create_templates
# License: Apache-2.0
#
# This repo contains create_templates_csv.py which does:
#   - Parse MMseqs2 alignment results
#   - Open matching CIF files from PDB_RNA/
#   - Align query sequence ato template sequence
#   - Copy C1' atom coordinates from template to template
#   - Handle insertions/deletions by interpolation
#   - Output templates.csv in competition-compatible format
# ============================================================
TEMPLATES_REPO = os.path.join(WORK_DIR, 'create_templates')

if not os.path.isdir(TEMPLATES_REPO):
    !git clone https://github.com/DasLab/create_templates.git {TEMPLATES_REPO}
    print(f'Cloned to {TEMPLATES_REPO}')
else:
    print(f'Already cloned at {TEMPLATES_REPO}')

# Verify the key script exists
script_path = os.path.join(TEMPLATES_REPO, 'create_templates_csv.py')
assert os.path.isfile(script_path), f'Script not found: {script_path}'
print(f'create_templates_csv.py: OK')

In [ ]:
# ============================================================
# CELL 5: Prepare query FASTA from test sequences
# ============================================================
# MMseqs2 requires input sequences in FASTA format.
# We convert test_sequences.csv → query.fasta
# ============================================================
QUERY_FASTA = os.path.join(WORK_DIR, 'query.fasta')

with open(QUERY_FASTA, 'w') as f:
    for _, row in test_df.iterrows():
        f.write(f'>{row["target_id"]}\n{row["sequence"]}\n')

# Verify
with open(QUERY_FASTA) as f:
    lines = f.readlines()
print(f'Query FASTA: {len(lines)//2} sequences written to {QUERY_FASTA}')
print(f'First entry:')
print(f'  {lines[0].strip()}')
print(f'  {lines[1].strip()[:60]}...')

In [ ]:
# ============================================================
# CELL 6: Run MMseqs2 search pipeline
# ============================================================
import time

QUERY_DB   = os.path.join(WORK_DIR, 'queryDB')
TARGET_DB  = os.path.join(WORK_DIR, 'targetDB')
RESULT_DB  = os.path.join(WORK_DIR, 'resultDB')
TMP_DIR    = os.path.join(WORK_DIR, 'mmseqs_tmp')
RESULT_TXT = os.path.join(WORK_DIR, 'Result.txt')

os.makedirs(TMP_DIR, exist_ok=True)

# Clean up old result if exists (prevents cached broken result)
import shutil
for f in [QUERY_DB, TARGET_DB, RESULT_DB]:
    for ext in ['', '.index', '.dbtype', '.lookup', '.source']:
        if os.path.exists(f + ext):
            os.remove(f + ext)

# Step 1: Build query database
print('Step 1/4: Building query database...')
t0 = time.time()
!mmseqs createdb {QUERY_FASTA} {QUERY_DB} --dbtype 2
print(f'  Done ({time.time()-t0:.1f}s)')

# Step 2: Build target database from PDB RNA sequences
print('\nStep 2/4: Building target database from PDB RNA...')
t0 = time.time()
!mmseqs createdb {FASTA_DB} {TARGET_DB} --dbtype 2
print(f'  Done ({time.time()-t0:.1f}s)')

# Step 3: Search — NOTE: -a flag is required for convertalis to work
print('\nStep 3/4: Searching (this may take several minutes)...')
t0 = time.time()
!mmseqs search {QUERY_DB} {TARGET_DB} {RESULT_DB} {TMP_DIR} \
    -s 7.5 \
    --search-type 3 \
    -e 10 \
    -a
print(f'  Done ({time.time()-t0:.1f}s)')

# Step 4: Convert results to TSV
print('\nStep 4/4: Converting results...')
!mmseqs convertalis {QUERY_DB} {TARGET_DB} {RESULT_DB} {RESULT_TXT} \
    --format-output query,target,evalue,qstart,qend,tstart,tend,qaln,taln \
    --search-type 3

# Verify results
if os.path.isfile(RESULT_TXT):
    with open(RESULT_TXT) as f:
        hits = f.readlines()
    print(f'\nTotal MMseqs2 hits: {len(hits)}')

    queries_with_hits = set()
    for line in hits:
        queries_with_hits.add(line.split('\t')[0])
    print(f'Test targets with hits: {len(queries_with_hits)} / {len(test_df)}')

    if hits:
        print(f'\nFirst 3 hits:')
        for line in hits[:3]:
            parts = line.strip().split('\t')
            print(f'  Query={parts[0]:10s} Target={parts[1]:20s} E-value={parts[2]}')
else:
    print('ERROR: No results file. Check MMseqs2 output above for errors.')

In [ ]:
# ============================================================
# CELL 7: Run DasLab create_templates_csv.py
# ============================================================
TEMPLATES_CSV = os.path.join(WORK_DIR, 'templates.csv')

# Create the relative path the script incorrectly writes to
os.makedirs('/kaggle/working/create_templates/kaggle/working', exist_ok=True)

# Remove old templates.csv to force fresh copy
if os.path.isfile('/kaggle/working/templates.csv'):
    os.remove('/kaggle/working/templates.csv')
    print('Removed old templates.csv')

print('Running create_templates_csv.py...')
print(f'  Sequences:  {TEST_CSV}')
print(f'  MMseqs2:    {RESULT_TXT}')
print(f'  CIF dir:    {PDB_RNA_DIR}')
print(f'  Output:     {TEMPLATES_CSV}')
print()

t0 = time.time()
import subprocess
result = subprocess.run(
    ['python3', 'create_templates_csv.py',
     '-s', TEST_CSV,
     '--mmseqs_results_file', RESULT_TXT,
     '--cif_dir', PDB_RNA_DIR,
     '--outfile', TEMPLATES_CSV,
     '--max_templates', '5',
     '--skip_temporal_cutoff'],
    cwd=TEMPLATES_REPO,
    capture_output=True, text=True
)
print(result.stdout[-3000:])

elapsed = time.time() - t0
print(f'\nCompleted in {elapsed:.1f}s')

# Always overwrite — don't skip if file exists
wrong_path = '/kaggle/working/create_templates/kaggle/working/templates.csv'
right_path = '/kaggle/working/templates.csv'

if os.path.isfile(wrong_path):
    import shutil
    shutil.copy(wrong_path, right_path)
    print('Copied templates.csv to correct location')

if os.path.isfile(TEMPLATES_CSV):
    tmpl_df = pd.read_csv(TEMPLATES_CSV)
    print(f'Templates output: {tmpl_df.shape}')
    print(f'Columns: {list(tmpl_df.columns)[:10]}')
    # Verify real coordinates (not zero)
    sample = tmpl_df['x_1'].dropna().head(3).values
    print(f'x_1 sample (should NOT be 0.0): {sample}')
else:
    print('ERROR: templates.csv not generated. Check output above.')

In [ ]:
# ============================================================
# CELL 8: Convert templates.csv to submission.csv
# ============================================================
import numpy as np

SUBMISSION_CSV = os.path.join(WORK_DIR, 'submission.csv')

if os.path.isfile(TEMPLATES_CSV):
    tmpl_df = pd.read_csv(TEMPLATES_CSV)

    expected_cols = ['ID', 'resname', 'resid', 'x_1', 'y_1', 'z_1']
    if all(c in tmpl_df.columns for c in expected_cols):
        tmpl_df.to_csv(SUBMISSION_CSV, index=False)
        print(f'Output already in submission format.')
    else:
        print(f'Output columns: {list(tmpl_df.columns)}')
        tmpl_df.to_csv(SUBMISSION_CSV, index=False)

    print(f'\nSubmission: {SUBMISSION_CSV}')
    print(f'Shape: {tmpl_df.shape}')
else:
    print('No templates.csv found. Cannot generate submission.')

In [ ]:
# ============================================================
# CELL 9: Fill predictions with diversity (NO centering)
# ============================================================
import numpy as np

sub_df = pd.read_csv(SUBMISSION_CSV)

# Step 1: Fix NaN in prediction 1
for ax in ['x', 'y', 'z']:
    sub_df[f'{ax}_1'] = sub_df[f'{ax}_1'].fillna(0.0)

# Step 2: Fill predictions 2-5 with real diversity
np.random.seed(42)
for i in range(2, 6):
    for ax in ['x', 'y', 'z']:
        base = f'{ax}_1'
        col = f'{ax}_{i}'
        sub_df[col] = sub_df[base] + np.random.normal(0, 5.0, size=len(sub_df))

sub_df.to_csv(SUBMISSION_CSV, index=False)

# Validate
print(f"Shape: {sub_df.shape}")
print(f"Null values: {sub_df.isnull().sum().sum()}")
coord_cols = [c for c in sub_df.columns if c.startswith(('x_','y_','z_'))]
all_c = sub_df[coord_cols].values.flatten()
print(f"Coord range: [{all_c.min():.2f}, {all_c.max():.2f}]")
print(f"Coord mean: {all_c.mean():.2f}")

stds = []
for ax in ['x','y','z']:
    cols = [f'{ax}_{i}' for i in range(1,6)]
    stds.append(sub_df[cols].std(axis=1).mean())
print(f"Diversity std — x:{stds[0]:.2f}  y:{stds[1]:.2f}  z:{stds[2]:.2f}")

degen = (sub_df[[f'x_{i}' for i in range(1,6)]].nunique(axis=1)==1).sum()
print(f"Identical prediction rows: {degen}  (should be 0)")

print(f"\nFirst 3 rows:")
print(sub_df[['ID','x_1','y_1','z_1','x_2','y_2','z_2']].head(3).to_string())
print(f"\nSubmission ready at: {SUBMISSION_CSV}")